# ETF V2.1: Strict Signal Alignment and Volume A/B

This notebook is a fresh V2-based experiment. It uses only `ret5d`, weekly Top3,
and three predeclared feature sets: baseline, ret5-volume confirmation, and
trend-volume confirmation.

The experiment runs two independent liquidity contracts: 20m and 50m average
daily turnover. They are capacity A/Bs, not a feature A/B, and each threshold
exports its own panel, diagnostics, score rows, and model bundles.

Research and JoinQuant online scoring share one signal contract: signal-date ETF
universe, price history, liquidity filter, rank population, feature columns, and
missing-value fill policy. Labels use next-trading-day open to the open five
trading days later, matching the weekly Monday rebalance approximation.


In [ ]:
# =========================
# 0. Config
# =========================
import os
import gc
import json
import math
import pickle
import hashlib
import datetime
import numpy as np
import pandas as pd
from jqdata import *

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OUT_DIR = "etf_ml_v2_1_strict_alignment_volume_ab_r2_outputs"
PANEL_CSV_TEMPLATE = os.path.join(OUT_DIR, "etf_ml_v2_1_weekly_panel_{tag}.csv")
SCORE_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_score_panel.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_summary.csv")
YEARLY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_yearly.csv")
STABILITY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_rolling_stability.csv")
WEEKLY_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_weekly_signal_proxy.csv")
MANIFEST_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_model_manifest.csv")
ALIGNMENT_SNAPSHOT_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_alignment_snapshot.csv")
TOP20_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_score_top20.csv")
LATEST_TARGETS_CSV = os.path.join(OUT_DIR, "etf_ml_v2_1_latest_targets.csv")
PANEL_DIAGNOSTICS_CSV_TEMPLATE = os.path.join(OUT_DIR, "etf_ml_v2_1_panel_build_diagnostics_{tag}.csv")

REBUILD_DATA = True
START_DATE = "2021-01-01"
END_DATE = "2026-06-30"
LOOKBACK_DAYS = 60
LABEL_HORIZON_DAYS = 5
MIN_LISTING_DAYS = 180
# Two independent candidate-pool contracts. Never compare them as a pure feature A/B.
LIQUIDITY_SPECS = [("money2000w", 20000000.0), ("money5000w", 50000000.0)]
ETF_CHUNK_SIZE = 120
STOCK_NUM = 3
BENCHMARK = "000985.XSHG"
RANDOM_SEED = 42
SCORE_TOP_N = 20
COST_RATE = 0.0001

TREND_WINDOWS = [10, 20, 25, 60]
TREND_WEIGHT_END = 2.0
PRICE_HISTORY_COUNT = max(LOOKBACK_DAYS + 1, max(TREND_WINDOWS) + 1)
POOL_CONTEXT_WINDOW = 25
RESEARCH_VERSION = "etf_ml_v2_1_strict_alignment_volume_ab_r2"

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单",
]
ETF_FALLBACK_PREFIXES = ("510", "511", "512", "513", "515", "516", "517", "518", "519", "520", "560", "561", "562", "563", "588", "589", "159")

ROLLING_OOS_SPECS = [
    ("rolling3y_test2024", "2021-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
    ("rolling3y_test2025", "2022-01-01", "2024-12-31", "2025-01-01", "2025-12-31"),
    ("rolling3y_test2026", "2023-01-01", "2025-12-31", "2026-01-01", "2026-06-30"),
]
# This is a sensitivity layer, not part of primary model selection.
RUN_EXPANDING_SENSITIVITY = False
EXPANDING_SENSITIVITY_SPECS = [
    ("expanding_test2025", "2021-01-01", "2024-12-31", "2025-01-01", "2025-12-31"),
    ("expanding_test2026", "2021-01-01", "2025-12-31", "2026-01-01", "2026-06-30"),
]
EXPERIMENT_SPECS = list(ROLLING_OOS_SPECS)
if RUN_EXPANDING_SENSITIVITY:
    EXPERIMENT_SPECS.extend(EXPANDING_SENSITIVITY_SPECS)
TARGET_SPECS = [("ret5d", "future_ret_5d")]

BASE_PRICE_FEATURE_COLS = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "vol_5", "vol_20", "vol_60",
    "close_to_ma20", "close_to_ma60", "ma5_to_ma20", "ma20_to_ma60",
    "drawdown_20", "drawdown_60", "amp_20", "amp_60",
    "money_mean_20", "money_ratio_5_20", "money_ratio_20_60",
    "volume_ratio_5_20", "volume_ratio_20_60", "max_ret_20", "min_ret_20",
]
TREND_FEATURE_COLS = []
for _window in TREND_WINDOWS:
    TREND_FEATURE_COLS.extend([
        "trend_ann_%s" % _window,
        "trend_r2_%s" % _window,
        "trend_score_%s" % _window,
        "trend_vol_%s" % _window,
        "trend_score_vol_adj_%s" % _window,
        "trend_simple_ann_%s" % _window,
    ])
CONTEXT_FEATURE_COLS = [
    "pool_breadth_%s" % POOL_CONTEXT_WINDOW,
    "pool_median_vol_%s" % POOL_CONTEXT_WINDOW,
]
RAW_FEATURE_COLS = BASE_PRICE_FEATURE_COLS + TREND_FEATURE_COLS
RANK_FEATURE_COLS = ["rank_" + col for col in RAW_FEATURE_COLS]
BASELINE_FEATURE_COLS = RAW_FEATURE_COLS + RANK_FEATURE_COLS + CONTEXT_FEATURE_COLS
VOLUME_AB_FEATURES = {
    "plus_ret5_volume_confirm": "ret_5_x_rank_money_ratio_5_20",
    "plus_trend20_volume_confirm": "trend_score_20_x_rank_money_ratio_5_20",
}
FEATURE_SETS = {"baseline": list(BASELINE_FEATURE_COLS)}
for _name, _col in VOLUME_AB_FEATURES.items():
    FEATURE_SETS[_name] = list(BASELINE_FEATURE_COLS) + [_col]
ALL_FEATURE_COLS = list(BASELINE_FEATURE_COLS) + list(VOLUME_AB_FEATURES.values())

LGB_PARAMS = {
    "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
    "learning_rate": 0.04, "num_leaves": 31, "max_depth": 5,
    "min_data_in_leaf": 80, "feature_fraction": 0.90,
    "bagging_fraction": 0.85, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 1.0,
    "verbose": -1, "seed": RANDOM_SEED,
}
NUM_BOOST_ROUND = 180

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
print("V2.1 baseline features:", len(BASELINE_FEATURE_COLS))
print("V2.1 feature sets:", {k: len(v) for k, v in FEATURE_SETS.items()})
print("V2.1 rolling OOS specs:", EXPERIMENT_SPECS)


In [ ]:
# =========================
# 1. Signal-date data and feature contract
# =========================
PANEL_BUILD_DIAGNOSTICS = []
ETF_UNIVERSE_SOURCE_BY_DATE = {}
ETF_MASTER_DF = None
ETF_MASTER_SOURCE = ""


def record_panel_build_diagnostic(diag, status):
    diag["status"] = status
    PANEL_BUILD_DIAGNOSTICS.append(diag)


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def get_weekly_feature_dates(start_date, end_date):
    days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(days) == 0:
        return []
    values = pd.Series(days)
    return [pd.Timestamp(gdf.max()).normalize() for _, gdf in values.groupby(values.dt.strftime("%Y-%W"))]


def should_exclude_name(name):
    text = str(name)
    return any(keyword and keyword in text for keyword in EXCLUDE_NAME_KEYWORDS)


def is_exchange_etf_code(code):
    return str(code).split(".")[0].startswith(ETF_FALLBACK_PREFIXES)


def load_etf_master():
    global ETF_MASTER_DF, ETF_MASTER_SOURCE
    if ETF_MASTER_DF is not None:
        return ETF_MASTER_DF
    query_report = []
    for source_name, security_types in [("etf_master", ["etf"]), ("fund_master_fallback", ["fund"])]:
        try:
            candidate = get_all_securities(security_types)
            row_count = 0 if candidate is None else len(candidate)
            query_report.append("%s=%s" % (source_name, row_count))
        except Exception as err:
            query_report.append("%s=ERR:%s" % (source_name, err))
            continue
        if candidate is not None and not candidate.empty:
            ETF_MASTER_DF = candidate.copy()
            ETF_MASTER_SOURCE = source_name
            ETF_MASTER_DF["_query_report"] = ";".join(query_report)
            print("ETF master loaded:", ETF_MASTER_SOURCE, ETF_MASTER_DF.shape)
            return ETF_MASTER_DF
    raise RuntimeError("ETF master unavailable: " + ";".join(query_report))


def get_etf_universe_on_signal_date(signal_date):
    signal_ts = pd.Timestamp(signal_date).normalize()
    date_text = signal_ts.strftime("%Y-%m-%d")
    sec_df = load_etf_master()
    query_report = str(sec_df["_query_report"].iloc[0]) if "_query_report" in sec_df.columns else ""
    out, skipped_missing_dates = [], 0
    for code, row in sec_df.iterrows():
        try:
            # fund fallback can include LOFs/open-end funds; keep exchange ETF code ranges only.
            if ETF_MASTER_SOURCE.startswith("fund"):
                item_type = str(row.get("type", "")).lower()
                if item_type and item_type != "etf" and not is_exchange_etf_code(code):
                    continue
            start_date = row.get("start_date", None)
            end_date = row.get("end_date", None)
            if pd.isnull(start_date):
                skipped_missing_dates += 1
                continue
            start_ts = pd.Timestamp(start_date).normalize()
            end_ts = pd.Timestamp(end_date).normalize() if not pd.isnull(end_date) else pd.Timestamp("2200-01-01")
            if start_ts + pd.Timedelta(days=MIN_LISTING_DAYS) > signal_ts or end_ts < signal_ts:
                continue
            if should_exclude_name(row.get("display_name", "")):
                continue
            out.append(code)
        except Exception:
            skipped_missing_dates += 1
    ETF_UNIVERSE_SOURCE_BY_DATE[date_text] = {
        "source": ETF_MASTER_SOURCE + "_start_end_filtered",
        "query_report": query_report, "master_count": len(sec_df),
        "filtered_count": len(out), "skipped_missing_dates": skipped_missing_dates,
    }
    if len(out) == 0:
        print("ETF master filtered empty", date_text, ETF_UNIVERSE_SOURCE_BY_DATE[date_text])
    return out


def safe_ratio(a, b):
    if pd.isnull(a) or pd.isnull(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def safe_ret(close, days):
    if len(close) <= days:
        return np.nan
    base = close.iloc[-days - 1]
    if pd.isnull(base) or base <= 0:
        return np.nan
    return close.iloc[-1] / base - 1.0


def calc_trend_metrics(close, days):
    if len(close) <= days:
        return dict.fromkeys(["ann", "r2", "score", "vol", "score_vol_adj", "simple_ann"], np.nan)
    recent = pd.Series(close.iloc[-(days + 1):].astype(float).values)
    if recent.isnull().any() or (recent <= 0).any():
        return dict.fromkeys(["ann", "r2", "score", "vol", "score_vol_adj", "simple_ann"], np.nan)
    y = np.log(recent.values)
    x = np.arange(len(y))
    weights = np.linspace(1.0, TREND_WEIGHT_END, len(y))
    slope, intercept = np.polyfit(x, y, 1, w=weights)
    ann = math.exp(slope * 250.0) - 1.0
    fit = slope * x + intercept
    ss_res = np.sum(weights * (y - fit) ** 2)
    ss_tot = np.sum(weights * (y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else 0.0
    r2 = max(0.0, min(1.0, float(r2)))
    daily_returns = recent.pct_change().dropna()
    vol = float(daily_returns.std() * math.sqrt(250.0)) if len(daily_returns) > 1 else np.nan
    period_ret = recent.iloc[-1] / recent.iloc[0] - 1.0
    simple_ann = (1.0 + period_ret) ** (250.0 / float(days)) - 1.0 if 1.0 + period_ret > 0 else np.nan
    score = ann * r2
    score_vol_adj = score * 0.20 / max(vol, 0.05) if not pd.isnull(vol) else np.nan
    return {"ann": ann, "r2": r2, "score": score, "vol": vol, "score_vol_adj": score_vol_adj, "simple_ann": simple_ann}


def calc_one_etf_features(code, price_df):
    df = price_df.sort_values("time").copy()
    if len(df) < max(40, int(LOOKBACK_DAYS * 0.8)):
        return None
    required = ["open", "high", "low", "close", "volume", "money"]
    if any(col not in df.columns for col in required):
        return None
    close = pd.Series(df["close"].astype(float).values)
    high = pd.Series(df["high"].astype(float).values)
    low = pd.Series(df["low"].astype(float).values)
    volume = pd.Series(df["volume"].astype(float).values)
    money = pd.Series(df["money"].astype(float).values)
    if close.isnull().any() or close.iloc[-1] <= 0:
        return None
    ret = close.pct_change()
    rec = {"code": code}
    for days in [1, 5, 10, 20, 60]:
        rec["ret_%s" % days] = safe_ret(close, days)
    for days in [5, 20, 60]:
        rec["vol_%s" % days] = ret.tail(days).std()
    rec["close_to_ma20"] = safe_ratio(close.iloc[-1], close.tail(20).mean()) - 1.0
    rec["close_to_ma60"] = safe_ratio(close.iloc[-1], close.tail(60).mean()) - 1.0
    rec["ma5_to_ma20"] = safe_ratio(close.tail(5).mean(), close.tail(20).mean()) - 1.0
    rec["ma20_to_ma60"] = safe_ratio(close.tail(20).mean(), close.tail(60).mean()) - 1.0
    rec["drawdown_20"] = safe_ratio(close.iloc[-1], close.tail(20).max()) - 1.0
    rec["drawdown_60"] = safe_ratio(close.iloc[-1], close.tail(60).max()) - 1.0
    rec["amp_20"] = (high.tail(20) / low.tail(20) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["amp_60"] = (high.tail(60) / low.tail(60) - 1.0).replace([np.inf, -np.inf], np.nan).mean()
    rec["money_mean_20"] = money.tail(20).mean()
    rec["money_ratio_5_20"] = safe_ratio(money.tail(5).mean(), money.tail(20).mean()) - 1.0
    rec["money_ratio_20_60"] = safe_ratio(money.tail(20).mean(), money.tail(60).mean()) - 1.0
    rec["volume_ratio_5_20"] = safe_ratio(volume.tail(5).mean(), volume.tail(20).mean()) - 1.0
    rec["volume_ratio_20_60"] = safe_ratio(volume.tail(20).mean(), volume.tail(60).mean()) - 1.0
    rec["max_ret_20"] = ret.tail(20).max()
    rec["min_ret_20"] = ret.tail(20).min()
    for window in TREND_WINDOWS:
        metrics = calc_trend_metrics(close, window)
        for key, value in metrics.items():
            rec["trend_%s_%s" % (key, window)] = value
    return rec


def fetch_feature_rows(signal_date, etfs):
    rows = []
    fields = ["open", "high", "low", "close", "volume", "money"]
    for one_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        price_df = get_price(one_chunk, end_date=signal_date, frequency="daily", fields=fields,
                             count=PRICE_HISTORY_COUNT, panel=False, fq="pre", skip_paused=False)
        if price_df is None or price_df.empty:
            continue
        for code, one in price_df.groupby("code"):
            rec = calc_one_etf_features(code, one)
            if rec is not None:
                rows.append(rec)
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan) if rows else pd.DataFrame()


def fetch_future_returns(signal_date, etfs):
    # Friday signal -> next trading-day open entry -> five trading days later open exit.
    # This matches the Monday 09:35 weekly rebalance much better than close-to-close labels.
    days = pd.to_datetime(get_trade_days(start_date=signal_date, count=LABEL_HORIZON_DAYS + 2))
    columns = ["code", "future_ret_5d", "execution_date", "next_date"]
    if len(days) < LABEL_HORIZON_DAYS + 2:
        return pd.DataFrame(columns=columns)
    execution_date = pd.Timestamp(days[1]).normalize()
    next_date = pd.Timestamp(days[-1]).normalize()
    rows = []
    for one_chunk in chunks(etfs, ETF_CHUNK_SIZE):
        px = get_price(one_chunk, start_date=execution_date, end_date=next_date, frequency="daily",
                       fields=["open"], panel=False, fq="pre", skip_paused=False)
        if px is None or px.empty:
            continue
        for code, one in px.groupby("code"):
            one = one.sort_values("time")
            if len(one) < 2 or float(one.iloc[0]["open"]) <= 0:
                continue
            rows.append({
                "code": code,
                "future_ret_5d": float(one.iloc[-1]["open"]) / float(one.iloc[0]["open"]) - 1.0,
                "execution_date": execution_date,
                "next_date": next_date,
            })
    return pd.DataFrame(rows, columns=columns)


def add_pool_context_features(df):
    out = df.copy()
    out["pool_breadth_%s" % POOL_CONTEXT_WINDOW] = float((out["trend_ann_%s" % POOL_CONTEXT_WINDOW] > 0).mean())
    out["pool_median_vol_%s" % POOL_CONTEXT_WINDOW] = float(out["trend_vol_%s" % POOL_CONTEXT_WINDOW].median())
    return out


def add_rank_features(df):
    out = df.copy()
    for col in RAW_FEATURE_COLS:
        out["rank_" + col] = out[col].rank(pct=True, method="average")
    return out


def add_volume_ab_features(df):
    out = df.copy()
    out["ret_5_x_rank_money_ratio_5_20"] = out["ret_5"] * out["rank_money_ratio_5_20"]
    out["trend_score_20_x_rank_money_ratio_5_20"] = out["trend_score_20"] * out["rank_money_ratio_5_20"]
    return out.replace([np.inf, -np.inf], np.nan)


def hash_lines(parts):
    return hashlib.sha256("\n".join(parts).encode("utf-8")).hexdigest()


def feature_snapshot(signal_date, raw_universe, feature_df):
    universe = sorted(set(str(code) for code in raw_universe))
    ordered = feature_df.loc[:, ["code"] + ALL_FEATURE_COLS].copy().sort_values("code")
    feature_lines = []
    for _, row in ordered.iterrows():
        values = ["NA" if pd.isnull(row[col]) else "%.10f" % float(row[col]) for col in ALL_FEATURE_COLS]
        feature_lines.append("|".join([str(row["code"])] + values))
    return {
        "signal_date": str(pd.Timestamp(signal_date).date()),
        "raw_universe_count": len(universe),
        "feature_pool_count": len(ordered),
        "raw_universe_hash": hash_lines(universe),
        "feature_pool_hash": hash_lines(ordered["code"].astype(str).tolist()),
        "feature_hash": hash_lines(feature_lines),
    }


def build_one_feature_date(signal_date, liquidity_tag, min_avg_money_20):
    signal_date = pd.Timestamp(signal_date).normalize()
    diag = {
        "signal_date": signal_date, "liquidity_tag": liquidity_tag, "universe_source": "",
        "min_avg_money_20": float(min_avg_money_20), "raw_universe_count": 0,
        "feature_row_count": 0, "liquid_pool_count": 0,
        "label_row_count": 0, "merged_row_count": 0,
    }
    raw_universe = get_etf_universe_on_signal_date(signal_date)
    source_info = ETF_UNIVERSE_SOURCE_BY_DATE.get(signal_date.strftime("%Y-%m-%d"), {})
    diag["universe_source"] = source_info.get("source", "unknown")
    diag["universe_query_report"] = source_info.get("query_report", "")
    diag["raw_universe_count"] = len(raw_universe)
    if len(raw_universe) == 0:
        record_panel_build_diagnostic(diag, "empty_raw_universe")
        return pd.DataFrame()
    feature_df = fetch_feature_rows(signal_date, raw_universe)
    diag["feature_row_count"] = len(feature_df)
    if feature_df.empty:
        record_panel_build_diagnostic(diag, "empty_price_features")
        return pd.DataFrame()
    feature_df = feature_df[feature_df["money_mean_20"] >= float(min_avg_money_20)].copy()
    diag["liquid_pool_count"] = len(feature_df)
    if feature_df.empty:
        record_panel_build_diagnostic(diag, "empty_after_liquidity")
        return pd.DataFrame()
    feature_df = add_pool_context_features(feature_df)
    feature_df = add_rank_features(feature_df)
    feature_df = add_volume_ab_features(feature_df)
    snapshot = feature_snapshot(signal_date, raw_universe, feature_df)
    label_df = fetch_future_returns(signal_date, feature_df["code"].tolist())
    diag["label_row_count"] = len(label_df)
    if label_df.empty:
        record_panel_build_diagnostic(diag, "empty_execution_labels")
        return pd.DataFrame()
    out = feature_df.merge(label_df, on="code", how="inner")
    diag["merged_row_count"] = len(out)
    if out.empty:
        record_panel_build_diagnostic(diag, "empty_after_label_merge")
        return out
    out["liquidity_tag"] = liquidity_tag
    out["min_avg_money_20"] = float(min_avg_money_20)
    out["universe_source"] = diag["universe_source"]
    out["universe_query_report"] = diag["universe_query_report"]
    out["signal_date"] = signal_date
    out["feature_date"] = signal_date
    for key, value in snapshot.items():
        out[key] = value
    record_panel_build_diagnostic(diag, "ok")
    return out


def assert_feature_contract(panel):
    missing = [col for col in ALL_FEATURE_COLS if col not in panel.columns]
    if missing:
        raise AssertionError("missing V2.1 feature columns: %s" % missing)
    for name, cols in FEATURE_SETS.items():
        if len(cols) != len(set(cols)) or any("future" in col or "target" in col for col in cols):
            raise AssertionError("invalid feature set: %s" % name)
    return True


In [ ]:
# =========================
# 2. Build or load point-in-time panel
# =========================
def build_weekly_panel(liquidity_tag, min_avg_money_20, diagnostics_csv):
    del PANEL_BUILD_DIAGNOSTICS[:]
    dates = get_weekly_feature_dates(START_DATE, END_DATE)
    print("build", liquidity_tag, "threshold=%.0f" % min_avg_money_20, "weeks:", len(dates))
    parts = []
    for signal_date in tqdm(dates, desc="build V2.1 %s" % liquidity_tag):
        one = build_one_feature_date(signal_date, liquidity_tag, min_avg_money_20)
        if one is not None and not one.empty:
            parts.append(one)
        if len(PANEL_BUILD_DIAGNOSTICS) % 20 == 0:
            gc.collect()
    diagnostics_df = pd.DataFrame(PANEL_BUILD_DIAGNOSTICS)
    diagnostics_df.to_csv(diagnostics_csv, index=False)
    if not diagnostics_df.empty:
        print("panel build status", liquidity_tag)
        print(diagnostics_df["status"].value_counts())
        print(diagnostics_df[["raw_universe_count", "feature_row_count", "liquid_pool_count", "label_row_count", "merged_row_count"]].describe())
    if not parts:
        raise RuntimeError("no ETF panel rows built for " + liquidity_tag + "; inspect " + diagnostics_csv)
    panel = pd.concat(parts, ignore_index=True, sort=False)
    quality = panel.groupby("feature_date")["future_ret_5d"].agg(["count", lambda s: (s == 0).mean()])
    bad_dates = set(quality[quality.iloc[:, 1] >= 0.98].index)
    if bad_dates:
        panel = panel[~panel["feature_date"].isin(bad_dates)].copy()
    for col in ["signal_date", "feature_date", "execution_date", "next_date"]:
        if col in panel.columns:
            panel[col] = pd.to_datetime(panel[col])
    assert_feature_contract(panel)
    return panel


PANEL_PATHS = {}
alignment_parts = []
for liquidity_tag, min_avg_money_20 in LIQUIDITY_SPECS:
    panel_csv = PANEL_CSV_TEMPLATE.format(tag=liquidity_tag)
    diagnostics_csv = PANEL_DIAGNOSTICS_CSV_TEMPLATE.format(tag=liquidity_tag)
    if REBUILD_DATA or not os.path.exists(panel_csv):
        panel_df = build_weekly_panel(liquidity_tag, min_avg_money_20, diagnostics_csv)
        panel_df.to_csv(panel_csv, index=False)
    else:
        panel_df = pd.read_csv(panel_csv)
        for col in ["signal_date", "feature_date", "execution_date", "next_date"]:
            if col in panel_df.columns:
                panel_df[col] = pd.to_datetime(panel_df[col])
        assert_feature_contract(panel_df)
    PANEL_PATHS[liquidity_tag] = panel_csv
    snapshot = panel_df.sort_values("feature_date").drop_duplicates("feature_date")[
        ["liquidity_tag", "min_avg_money_20", "universe_source", "feature_date", "signal_date", "raw_universe_count", "feature_pool_count", "raw_universe_hash", "feature_pool_hash", "feature_hash"]
    ].copy()
    alignment_parts.append(snapshot)
    print("panel", liquidity_tag, panel_df.shape, "saved:", panel_csv)
    del panel_df, snapshot
    gc.collect()
alignment_snapshot_df = pd.concat(alignment_parts, ignore_index=True, sort=False)
alignment_snapshot_df.to_csv(ALIGNMENT_SNAPSHOT_CSV, index=False)
print("alignment snapshots:", alignment_snapshot_df.shape)
print("panel paths:", PANEL_PATHS)
display(alignment_snapshot_df.head())


In [ ]:
# =========================
# 3. Train ret5d models and export strict bundles
# =========================
META_COLS = ["code", "feature_date", "execution_date", "next_date", "future_ret_5d"]
SCORE_EXPORT_COLS = META_COLS + [
    "score", "model_name", "model_file", "liquidity_tag", "min_avg_money_20", "feature_set_name", "train_tag",
    "evaluation_protocol", "oos_start_date", "oos_end_date", "feature_spec_hash",
]


def feature_spec_hash(feature_set_name, feature_cols, liquidity_tag, min_avg_money_20):
    payload = {
        "research_version": RESEARCH_VERSION,
        "feature_set_name": feature_set_name,
        "feature_cols": list(feature_cols),
        "raw_feature_cols": list(RAW_FEATURE_COLS),
        "all_alignment_feature_cols": list(ALL_FEATURE_COLS),
        "lookback_days": LOOKBACK_DAYS,
        "price_history_count": PRICE_HISTORY_COUNT,
        "min_listing_days": MIN_LISTING_DAYS,
        "liquidity_tag": liquidity_tag, "min_avg_money_20": float(min_avg_money_20),
        "exclude_name_keywords": EXCLUDE_NAME_KEYWORDS,
        "signal_timing": "previous_completed_trading_day",
        "rank_population": "signal_date_feature_pool_after_liquidity_before_label_merge",
    }
    text = json.dumps(payload, sort_keys=True, ensure_ascii=True, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def clean_train_df(df, feature_cols):
    cols = META_COLS + list(feature_cols)
    out = df.loc[:, cols].replace([np.inf, -np.inf], np.nan)
    return out[~out["future_ret_5d"].isnull()].copy()


def train_lightgbm(train_df, feature_cols):
    try:
        import lightgbm as lgb
    except Exception as err:
        raise RuntimeError("LightGBM is required for V2.1; refusing model-backend fallback: %s" % err)
    x_raw = train_df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    fill_values = x_raw.median().to_dict()
    x = x_raw.fillna(pd.Series(fill_values)).fillna(0)
    dtrain = lgb.Dataset(x[feature_cols], label=train_df["future_ret_5d"].astype(float).values,
                         feature_name=list(feature_cols), free_raw_data=True)
    model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
    return model, fill_values


def predict_model(model, fill_values, df, feature_cols):
    x = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    x = x.fillna(pd.Series(fill_values)).fillna(0)
    return np.asarray(model.predict(x[feature_cols])).reshape(-1).astype(float)


def make_bundle(model, fill_values, feature_set_name, feature_cols, liquidity_tag, min_avg_money_20, train_tag, train_start, train_end, oos_start, oos_end):
    spec_hash = feature_spec_hash(feature_set_name, feature_cols, liquidity_tag, min_avg_money_20)
    return {
        "objective": "etf_ml_weekly_lgb_v2_1", "research_version": RESEARCH_VERSION,
        "model": model, "model_backend": "lightgbm", "target_name": "ret5d",
        "target_col": "future_ret_5d", "feature_set_name": feature_set_name,
        "feature_cols": list(feature_cols), "raw_feature_cols": list(RAW_FEATURE_COLS),
        "rank_feature_cols": list(RANK_FEATURE_COLS), "context_feature_cols": list(CONTEXT_FEATURE_COLS),
        "alignment_feature_cols": list(ALL_FEATURE_COLS), "feature_spec_hash": spec_hash,
        "fill_values": dict(fill_values), "lookback_days": LOOKBACK_DAYS,
        "trend_windows": list(TREND_WINDOWS), "trend_weight_end": TREND_WEIGHT_END,
        "price_history_count": PRICE_HISTORY_COUNT, "pool_context_window": POOL_CONTEXT_WINDOW,
        "label_horizon_days": LABEL_HORIZON_DAYS, "min_listing_days": MIN_LISTING_DAYS,
        "liquidity_tag": liquidity_tag, "min_avg_money_20": float(min_avg_money_20), "exclude_name_keywords": list(EXCLUDE_NAME_KEYWORDS),
        "stock_num": STOCK_NUM, "max_etf_group_holdings": 0, "selection_contract": "raw_top3_no_theme_cap",
        "rebalance_mode": "weekly", "rebalance_interval_weeks": 1, "benchmark": BENCHMARK, "train_tag": train_tag,
        "evaluation_protocol": "rolling_3y_next_year" if train_tag.startswith("rolling3y") else "expanding_sensitivity",
        "train_start": train_start, "train_end": train_end,
        "oos_start_date": oos_start, "oos_end_date": oos_end,
        "signal_contract": "Friday signal-date universe; Monday-open entry to next-Monday-open label; liquidity then context/rank/volume features; raw top3 selection",
    }


for path in [SCORE_CSV, MANIFEST_CSV]:
    if os.path.exists(path):
        os.remove(path)
score_parts, manifest_rows = [], []
for liquidity_tag, min_avg_money_20 in tqdm(LIQUIDITY_SPECS, desc="V2.1 liquidity pools"):
    panel_df = pd.read_csv(PANEL_PATHS[liquidity_tag])
    for col in ["feature_date", "execution_date", "next_date"]:
        panel_df[col] = pd.to_datetime(panel_df[col])
    assert_feature_contract(panel_df)
    for train_tag, train_start, train_end, oos_start, oos_end in tqdm(EXPERIMENT_SPECS, desc=liquidity_tag, leave=False):
        train_end_ts = pd.Timestamp(train_end)
        oos_start_ts = pd.Timestamp(oos_start)
        oos_end_ts = pd.Timestamp(oos_end)
        train_mask = (panel_df["feature_date"] >= pd.Timestamp(train_start)) & (panel_df["next_date"] <= train_end_ts)
        score_mask = (panel_df["feature_date"] >= oos_start_ts) & (panel_df["next_date"] <= oos_end_ts)
        score_base = panel_df.loc[score_mask, META_COLS + ALL_FEATURE_COLS].copy()
        if score_base.empty:
            raise RuntimeError("empty OOS score set: %s %s" % (liquidity_tag, train_tag))
        for feature_set_name, feature_cols in FEATURE_SETS.items():
            train_df = clean_train_df(panel_df.loc[train_mask], feature_cols)
            if train_df.empty:
                raise RuntimeError("empty training set: %s %s" % (liquidity_tag, train_tag))
            model, fill_values = train_lightgbm(train_df, feature_cols)
            bundle = make_bundle(model, fill_values, feature_set_name, feature_cols, liquidity_tag, min_avg_money_20, train_tag, train_start, train_end, oos_start, oos_end)
            model_name = "etf_ml_v2_1_lgb_ret5d_%s_%s_%s" % (liquidity_tag, feature_set_name, train_tag)
            model_file = "model_%s.pkl" % model_name
            with open(os.path.join(OUT_DIR, model_file), "wb") as handle:
                pickle.dump(bundle, handle, protocol=2)
            scored = score_base.loc[:, META_COLS].copy()
            scored["score"] = predict_model(model, fill_values, score_base, feature_cols)
            scored["model_name"] = model_name
            scored["model_file"] = model_file
            scored["liquidity_tag"] = liquidity_tag
            scored["min_avg_money_20"] = float(min_avg_money_20)
            scored["feature_set_name"] = feature_set_name
            scored["train_tag"] = train_tag
            scored["evaluation_protocol"] = bundle["evaluation_protocol"]
            scored["oos_start_date"] = oos_start
            scored["oos_end_date"] = oos_end
            scored["feature_spec_hash"] = bundle["feature_spec_hash"]
            scored = scored.loc[:, SCORE_EXPORT_COLS]
            score_parts.append(scored)
            manifest_rows.append({
                "model_name": model_name, "model_file": model_file, "liquidity_tag": liquidity_tag,
                "feature_set_name": feature_set_name, "feature_spec_hash": bundle["feature_spec_hash"],
                "train_tag": train_tag, "evaluation_protocol": bundle["evaluation_protocol"],
                "train_start": train_start, "train_end": train_end,
                "oos_start_date": oos_start, "oos_end_date": oos_end,
                "train_samples": len(train_df), "train_weeks": train_df["feature_date"].nunique(),
                "feature_count": len(feature_cols), "min_avg_money_20": float(min_avg_money_20),
            })
            del model, bundle, fill_values, train_df, scored
            gc.collect()
        del score_base
    del panel_df
    gc.collect()
score_df = pd.concat(score_parts, ignore_index=True, sort=False)
manifest_df = pd.DataFrame(manifest_rows)
score_df.to_csv(SCORE_CSV, index=False)
manifest_df.to_csv(MANIFEST_CSV, index=False)
display(manifest_df)


In [ ]:
# =========================
# 4. Signal-quality A/B evaluation (not execution optimisation)
# =========================
def summarize_returns(values):
    ret = pd.Series(values).dropna().astype(float)
    if ret.empty:
        return {}
    nav = (1.0 + ret).cumprod()
    drawdown = nav / nav.cummax() - 1.0
    return {
        "weeks": len(ret), "cum_ret": float(nav.iloc[-1] - 1.0),
        "mean_weekly_ret": float(ret.mean()), "win_rate": float((ret > 0).mean()),
        "weekly_sharpe": float(ret.mean() / ret.std()) if ret.std() > 0 else np.nan,
        "max_drawdown": float(drawdown.min()),
    }


def summarize_relative_returns(portfolio_values, benchmark_values):
    portfolio = pd.Series(portfolio_values).dropna().astype(float)
    benchmark = pd.Series(benchmark_values).dropna().astype(float)
    if len(portfolio) == 0 or len(portfolio) != len(benchmark):
        return {}
    portfolio_nav = float((1.0 + portfolio).prod())
    benchmark_nav = float((1.0 + benchmark).prod())
    weekly_excess = portfolio.values - benchmark.values
    return {
        "universe_equal_weight_cum_ret": benchmark_nav - 1.0,
        "relative_nav_excess": portfolio_nav / benchmark_nav - 1.0,
        "mean_weekly_excess": float(np.mean(weekly_excess)),
        "positive_weekly_excess_rate": float((weekly_excess > 0).mean()),
    }


def build_stability_scorecard(yearly_data):
    if yearly_data.empty:
        return pd.DataFrame()
    rows = []
    primary = yearly_data[yearly_data["evaluation_protocol"] == "rolling_3y_next_year"].copy()
    primary["is_full_calendar_year"] = pd.to_datetime(primary["oos_end_date"]).dt.strftime("%m-%d") == "12-31"
    for (liquidity_tag, feature_set_name), one in primary.groupby(["liquidity_tag", "feature_set_name"]):
        active = pd.to_numeric(one["relative_nav_excess"], errors="coerce")
        full_active = active[one["is_full_calendar_year"]].dropna()
        rows.append({
            "liquidity_tag": liquidity_tag, "feature_set_name": feature_set_name,
            "primary_period_count": int(active.notnull().sum()),
            "full_calendar_year_count": int(len(full_active)),
            "partial_period_count": int((~one["is_full_calendar_year"]).sum()),
            "positive_full_year_count": int((full_active > 0).sum()),
            "positive_full_year_rate": float((full_active > 0).mean()) if len(full_active) else np.nan,
            "min_full_year_relative_nav_excess": float(full_active.min()) if len(full_active) else np.nan,
            "median_full_year_relative_nav_excess": float(full_active.median()) if len(full_active) else np.nan,
            "all_full_test_years_positive": bool((full_active > 0).all()) if len(full_active) else False,
        })
    return pd.DataFrame(rows).sort_values(["positive_full_year_count", "median_full_year_relative_nav_excess"], ascending=False)


weekly_rows, top20_parts = [], []
merged_scores = score_df.copy()
for (model_name, feature_date), one in tqdm(merged_scores.groupby(["model_name", "feature_date"]), desc="V2.1 signal proxy"):
    one = one.sort_values("score", ascending=False).copy()
    top = one.head(STOCK_NUM)
    top20_parts.append(one.head(SCORE_TOP_N))
    median_ret = float(one["future_ret_5d"].median())
    equal_weight_ret = float(one["future_ret_5d"].mean())
    weekly_rows.append({
        "model_name": model_name, "model_file": top["model_file"].iloc[0],
        "liquidity_tag": top["liquidity_tag"].iloc[0], "min_avg_money_20": top["min_avg_money_20"].iloc[0],
        "feature_set_name": top["feature_set_name"].iloc[0], "train_tag": top["train_tag"].iloc[0],
        "evaluation_protocol": top["evaluation_protocol"].iloc[0],
        "oos_start_date": top["oos_start_date"].iloc[0], "oos_end_date": top["oos_end_date"].iloc[0],
        "feature_date": feature_date, "next_date": top["next_date"].iloc[0],
        "targets": ",".join(top["code"].tolist()), "top3_ret_5d": float(top["future_ret_5d"].mean()),
        "universe_median_ret_5d": median_ret,
        "universe_equal_weight_ret_5d": equal_weight_ret,
        "excess_vs_median_5d": float(top["future_ret_5d"].mean() - median_ret),
        "weekly_excess_vs_equal_weight": float(top["future_ret_5d"].mean() - equal_weight_ret),
        "rank_ic": float(one["score"].rank().corr(one["future_ret_5d"].rank(), method="pearson")),
    })
weekly_df = pd.DataFrame(weekly_rows)
top20_df = pd.concat(top20_parts, ignore_index=True, sort=False)
summary_rows, yearly_rows = [], []
for model_name, one in weekly_df.groupby("model_name"):
    row = {"model_name": model_name}
    row.update(summarize_returns(one["top3_ret_5d"]))
    row.update(summarize_relative_returns(one["top3_ret_5d"], one["universe_equal_weight_ret_5d"]))
    row["mean_excess_vs_median_5d"] = float(one["excess_vs_median_5d"].mean())
    row["mean_rank_ic"] = float(one["rank_ic"].mean())
    row["positive_rank_ic_rate"] = float((one["rank_ic"] > 0).mean())
    row.update(one.iloc[0][["model_file", "liquidity_tag", "min_avg_money_20", "feature_set_name", "train_tag", "evaluation_protocol", "oos_start_date", "oos_end_date"]].to_dict())
    summary_rows.append(row)
    for year, year_df in one.groupby(pd.to_datetime(one["feature_date"]).dt.year):
        yearly = {"model_name": model_name, "year": int(year)}
        yearly.update(summarize_returns(year_df["top3_ret_5d"]))
        yearly.update(summarize_relative_returns(year_df["top3_ret_5d"], year_df["universe_equal_weight_ret_5d"]))
        yearly["mean_rank_ic"] = float(year_df["rank_ic"].mean())
        yearly.update(one.iloc[0][["liquidity_tag", "min_avg_money_20", "feature_set_name", "train_tag", "evaluation_protocol", "oos_start_date", "oos_end_date"]].to_dict())
        yearly_rows.append(yearly)
summary_df = pd.DataFrame(summary_rows).sort_values(["relative_nav_excess", "mean_rank_ic"], ascending=False)
yearly_df = pd.DataFrame(yearly_rows).sort_values(["model_name", "year"])
stability_df = build_stability_scorecard(yearly_df)
weekly_df.to_csv(WEEKLY_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
yearly_df.to_csv(YEARLY_CSV, index=False)
stability_df.to_csv(STABILITY_CSV, index=False)
top20_df.to_csv(TOP20_CSV, index=False)
display(summary_df)
display(yearly_df)
display(stability_df)


In [ ]:
# =========================
# 5. Latest targets and alignment handoff
# =========================
latest_rows = []
for model_name, one in score_df.groupby("model_name"):
    latest_date = one["feature_date"].max()
    selected = one[one["feature_date"] == latest_date].sort_values("score", ascending=False).head(STOCK_NUM)
    for rank, (_, row) in enumerate(selected.iterrows(), 1):
        latest_rows.append({"model_name": model_name, "liquidity_tag": row["liquidity_tag"], "min_avg_money_20": row["min_avg_money_20"], "feature_set_name": row["feature_set_name"],
                            "feature_date": latest_date, "rank": rank, "code": row["code"], "score": row["score"]})
latest_targets_df = pd.DataFrame(latest_rows)
latest_targets_df.to_csv(LATEST_TARGETS_CSV, index=False)
display(latest_targets_df.head(30))
print("Saved panels:", PANEL_PATHS)
print("Saved outputs:", SCORE_CSV, MANIFEST_CSV, ALIGNMENT_SNAPSHOT_CSV, TOP20_CSV, WEEKLY_CSV, SUMMARY_CSV, YEARLY_CSV, STABILITY_CSV)
print("Run the matching JoinQuant strategy with one exported pkl. Compare feature_date, raw_universe_hash, feature_pool_hash, feature_hash, then Top20 scores.")
